# 实验 15 — dlt + dbt 一个脚本端到端

**目标**：把 dlt 的 EL 和 dbt 的 T 串成一个 Python 入口。两种方式：

1. **`subprocess` 调用 `dbt` CLI**（最简单，最透明，本仓库 [orchestration.py](../orchestration.py) 用的就是这种）
2. **`dlt.helpers.dbt` runner**（dlt 自带的 dbt 集成，能把 dbt 当作 dlt 的下一阶段，共享 LoadInfo / state）

生产里两种都常见。`subprocess` 更受控、易调试；`dlt.helpers.dbt` 在 Dagster/Airflow 之外纯 Python 编排里挺顺手。

In [ ]:
# 方式 1：subprocess 跑 orchestration.py
import subprocess
r = subprocess.run(['uv','run','python','orchestration.py'],
                   capture_output=True, text=True, cwd='..')
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-800:])

In [ ]:
# 方式 2：dlt.helpers.dbt 在 Python 里直接跑 dbt 模型
# 先装 dbt 相关 helper（如果还没装）
import subprocess
subprocess.run(['uv','pip','install','dlt[dbt]'], capture_output=True, cwd='..')

In [ ]:
import os, dlt
from dlt.helpers.dbt import package_runner
from dlt_pipelines.ecb_rates import ecb_source

# 重新构建 pipeline，跑 ingest
pipe = dlt.pipeline(
    pipeline_name='ecb_rates_pipeline',
    destination=dlt.destinations.duckdb('../data/sandbox.duckdb'),
    dataset_name='raw_ecb',
)
load_info = pipe.run(ecb_source())
print(load_info)

In [ ]:
# 用 dlt.helpers.dbt 的 runner 触发 dbt build
# 注意：runner 会在隔离 venv 里跑 dbt，避免和外层 Python 环境冲突
from dlt.helpers.dbt import create_runner

runner = create_runner(
    venv=None,                   # 用当前环境
    package_location='../dbt_project',
    package_profiles_dir='../dbt_project',
    package_profile_name='dlt_dbt_sandbox',
    auto_full_refresh_when_out_of_sync=False,
)
results = runner.run_all()
for r in results:
    print(f"{r.node_name:40s} {r.status:10s} {r.message or ''}")

## 生产环境踩坑提示

- **失败传播**：dlt 跑成功不代表 dbt 也跑成功。任何 orchestration 入口都要 **检查两段都 OK 才退出 0**——`orchestration.py` 里通过 `RuntimeError` 实现。
- **dlt-dbt 共享 metadata**：可以把 `dlt.current.load_id()` 注入 dbt `vars`（`dbt build --vars '{load_id: "abc"}'`），让 dbt 模型只处理本次 load 写入的数据。
- **两套 venv？**：`dlt.helpers.dbt` 默认建一个隔离 venv 给 dbt 用，避免依赖冲突。本地 sandbox 可关掉（`venv=None`）。生产 Dockerfile 里通常一起装，不需要再隔离。
- **观测性**：把 `load_info.loads_ids` 和 dbt run 结果都打到结构化日志（JSON），方便 Datadog / Cloudwatch 抓 metrics。
- **重跑语义**：dlt 是幂等（merge + state），dbt 是声明式重建，所以整个 pipeline 重跑应该是“可靠重入”的——除了 incremental dbt 模型需要 `--full-refresh` 才会回到全量基线。

## 思考题

- 修改 [orchestration.py](../orchestration.py)，让它接受一个 `--target` 参数转给 dbt（`dbt build --target prod`）。
- 想象 dlt 一次写入 1000 万行，dbt 又重新全表扫描很贵。怎么用 dlt 提供的 `load_id` 让 dbt 只处理本次新写入的数据？（提示：dbt 模型里 `where _dlt_load_id = {{ var('load_id') }}`）
- 比较 `subprocess` 和 `dlt.helpers.dbt`：调试时哪个看 stack trace 更直接？做 retry / partial-rerun 时哪个更可控？